# GRU Networks

## Start with the problem: remember selectively without a separate cell

A gated recurrent unit (GRU) is a recurrent network that uses gates to control how a hidden representation changes. Like an LSTM, it can preserve useful information across steps; unlike an LSTM, it does not maintain a separate cell state.

This lesson derives the common GRU equations, works through a scalar transition, and compares design choices. Frameworks differ slightly in equation conventions, so always check the implementation when matching weights by hand.

## Read the update as a learned interpolation

A common GRU convention computes reset and update gates:

$$r_t=\sigma(W_r x_t+U_r h_{t-1}+b_r),\qquad z_t=\sigma(W_z x_t+U_z h_{t-1}+b_z).$$

The reset gate controls how much of the prior state contributes to a candidate:

$$\tilde h_t=\tanh(W_hx_t+U_h(r_t\odot h_{t-1})+b_h).$$

The update gate blends the old state and candidate:

$$h_t=(1-z_t)\odot h_{t-1}+z_t\odot\tilde h_t.$$

Here $\odot$ is elementwise multiplication. Under this convention, $z_t$ near zero mostly retains the old state; $z_t$ near one mostly replaces it. Some libraries define the complement as the update gate, yielding an algebraically equivalent-looking equation with different gate labels.

## Work a scalar transition

Let $h_{t-1}=0.6$, $r_t=0.5$, $z_t=0.25$, and candidate $\tilde h_t=-0.2$. Then the candidate blend uses $r_t h_{t-1}=0.3$ inside its learned affine transform. Once the candidate is calculated, the state update is $h_t=(1-0.25)(0.6)+0.25(-0.2)=0.4$.

Three quarters of the prior state remain and one quarter of the candidate is written. The gates are learned functions of the input and previous state; the values above are selected only to make the interpolation transparent.

## Compare a GRU with an LSTM

| Property | GRU | LSTM |
| --- | --- | --- |
| Persistent state | Hidden state only | Cell state and hidden state |
| Main controls | Reset and update gates | Forget, input, and output gates |
| Typical parameter count | Often fewer | Often more |

A GRU's smaller parameterization may be attractive when data or compute are limited. An LSTM's separate cell pathway gives another control over memory and exposure. Neither is universally more accurate or faster: dimensions, sequence lengths, kernels, and data can change the comparison.

## Practical details and limitations

GRUs are useful for sequence classification, sensor streams, and compact recurrent baselines. For variable-length batches, mask padding or use packed sequences. For sequence-to-sequence work, specify whether outputs at every time step or only a summary state feed the task head.

GRUs still process recurrent steps sequentially and can still experience difficult gradients. They do not remove the need for validation, careful split design, and comparisons against simpler baselines or attention-based models.

**Takeaway:** a GRU uses a reset gate to shape its candidate and an update gate to interpolate between old and new state. It trades the LSTM's separate cell state for a compact gated recurrence.